In [21]:
from typing import Optional
from core.db import engine
from sqlalchemy import select
from adapters.linkedin.models import JobObservation

def get_job_observations(engine, limit: Optional[int] = None):
    stmt = select(JobObservation)
    if limit:
            stmt = stmt.limit(limit)
    with engine.connect() as conn:
        return [dict(row) for row in conn.execute(stmt).mappings()]

def get_job_observations_by_ids(engine, ids: list[str]):
    stmt = select(JobObservation).where(JobObservation.job_id.in_(ids))
    with engine.connect() as conn:
        return [dict(row) for row in conn.execute(stmt).mappings()]


In [22]:
jobs = get_job_observations(engine)

In [17]:
# from collections import Counter
# word_counts = Counter()
# for job in jobs:
#     word_counts.update(job['description'].lower().split())

# word_counts.most_common(100)

In [25]:

from pydantic import BaseModel, Field
from typing import List, Optional
import instructor
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

class KeyTerms(BaseModel):
    text: List[str] = Field(description="List of keywords and phrases")


class JobAdParser:

    system_prompt = """You are an expert AI recruiter finding key terms in job descriptions.



    Rules:
    - Keywords include tools, languages, platforms, frameworks, etc. (e.g. AWS, Python, React, SQL, etc.)
    - Don't change the formatting of the text. for example don't change 'AWS' to 'Amazon Web Services' or vise versa.
    - Key Phrases are phrases that are not single words but rather multi-word expressions.
    - Key phrases are usually skills in the job description(e.g. train models, writing tests, implement pipelines, pipeline desgin, data quality management.)
    """

    def __init__(self, model_name):
        self.model_name = model_name


        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        )
        self.instructor_client = instructor.from_openai(client)

    def parse(self, job_desc):


        user_prompt = f"""Extract key terms from the job description below.

        Job description:
        {job_desc}
        """
        response = self.instructor_client.chat.completions.create(
        model=self.model_name,
        messages=[
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_model=KeyTerms,
        temperature = 0.0
        )
        return response

In [27]:
jobs = get_job_observations_by_ids(engine, ['4417568462'])
len(jobs)

1

In [28]:
parser = JobAdParser("gpt-4o-mini")
response = parser.parse(jobs[0]['description'])
print(response)

text=['AWS', 'Python', 'SQL', 'Pyspark', 'Databricks', 'Snowflake', 'Oracle', 'Stored Procs', 'data services', 'development', 'testing', 'deployment', 'operation', 'monitoring', 'refinement', 'manage incidents', 'apply fixes', 'resolve systematic issues', 'triage issues', 'identify and implement solutions', 'design build', 'implementation experience', 'Data Engineering pipelines', 'data solutions in Cloud', 'on-premises assets']


In [31]:
for t in response.text:
    print(t)

AWS
Python
SQL
Pyspark
Databricks
Snowflake
Oracle
Stored Procs
data services
development
testing
deployment
operation
monitoring
refinement
manage incidents
apply fixes
resolve systematic issues
triage issues
identify and implement solutions
design build
implementation experience
Data Engineering pipelines
data solutions in Cloud
on-premises assets
